### Setup and Installation

In [89]:
!pip install google-generativeai -q
!pip install python-dotenv -q
!pip install neo4j -q
!pip install numpy -q
!pip install pypdf -q

In [102]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv(override=True)

True

### Initialize Gemini LLM

In [91]:
import google.generativeai as genai

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Add it to the .env file.")

genai.configure(api_key=api_key)

def pick_text_model() -> str:
    preferred = [
        "models/gemini-2.5-flash",
        "models/gemini-2.5-pro",
        "models/gemini-1.5-flash-latest",
        "models/gemini-1.5-pro-latest",
    ]
    available = [
        model.name
        for model in genai.list_models()
        if "generateContent" in getattr(model, "supported_generation_methods", [])
    ]

    candidates = []
    for name in preferred + available:
        if name not in candidates:
            candidates.append(name)

    for name in candidates:
        try:
            test_model = genai.GenerativeModel(name)
            test_response = test_model.generate_content("Reply with: ok")
            if getattr(test_response, "text", ""):
                return name
        except Exception:
            continue

    raise RuntimeError("No usable Gemini text-generation model is available for this API key.")

TEXT_MODEL = pick_text_model()
llm_model = genai.GenerativeModel(TEXT_MODEL)

def ask_gemini(prompt: str) -> str:
    response = llm_model.generate_content(prompt)
    return (response.text or "").strip()

print(f"Using text model: {TEXT_MODEL}")

Using text model: models/gemini-2.5-flash


### Initialize Gemini Embedding Model

In [92]:
def pick_embedding_model() -> str:
    preferred = [
        "models/text-embedding-004",
        "models/embedding-001",
    ]
    available = [
        model.name
        for model in genai.list_models()
        if "embedContent" in getattr(model, "supported_generation_methods", [])
    ]
    for name in preferred:
        if name in available:
            return name
    if available:
        return available[0]
    raise RuntimeError("No Gemini embedding model is available for this API key.")

EMBEDDING_MODEL = pick_embedding_model()

def embed_text(text: str, task_type: str = "retrieval_document") -> list:
    response = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type=task_type,
    )
    return response["embedding"]

print(f"Using embedding model: {EMBEDDING_MODEL}")

Using embedding model: models/gemini-embedding-001


### Load Data from PDF Files

In [93]:
from pathlib import Path
from pypdf import PdfReader

ROOT_PATH = Path(r"D:\MyProjects\AI Project\knowledge_graph_rag")
pdf_paths = sorted(ROOT_PATH.rglob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in: {ROOT_PATH}")

text_parts = []
for pdf_path in pdf_paths:
    reader = PdfReader(str(pdf_path))
    for page in reader.pages:
        text_parts.append(page.extract_text() or "")

text = "\n".join(part for part in text_parts if part.strip())

if not text.strip():
    raise ValueError("PDF files were found, but no extractable text was detected.")

print(f"Loaded {len(pdf_paths)} PDF file(s) from {ROOT_PATH}")
print(f"Combined text length: {len(text)} characters")

Loaded 1 PDF file(s) from D:\MyProjects\AI Project\knowledge_graph_rag
Combined text length: 14041 characters


### Split Documents into Chunks

In [94]:
def split_text(content: str, chunk_size: int = 250, chunk_overlap: int = 30) -> list:
    chunks = []
    start = 0
    while start < len(content):
        end = start + chunk_size
        chunks.append(content[start:end])
        start += chunk_size - chunk_overlap
    return chunks

chunks = split_text(text, chunk_size=250, chunk_overlap=30)
len(chunks), chunks[0][:120]

(64,
 'SITE SPECIFI C FERTILIZER\nRECOMMENDATION (SSFR) USING\nPLANT TEST KIT\nDevelopment of Site Specific Fertilizer Recommendat')

### Graph Initialization and Transformation

In [103]:
from typing import Optional
from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import AuthError, ServiceUnavailable

# Reload .env with override so updated Neo4j credentials/URI are applied
load_dotenv(override=True)

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

missing_neo4j = [
    name for name, value in {
        "NEO4J_URI": NEO4J_URI,
        "NEO4J_USERNAME": NEO4J_USERNAME,
        "NEO4J_PASSWORD": NEO4J_PASSWORD,
    }.items() if not value
]
if missing_neo4j:
    raise ValueError(f"Missing Neo4j config in .env: {', '.join(missing_neo4j)}")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

# Fail fast with a clear message if DNS/network/auth is not valid
try:
    with driver.session() as session:
        session.run("RETURN 1 AS ok").single()
    print(f"Neo4j connected: {NEO4J_URI}")
except AuthError as e:
    raise RuntimeError(
        f"Neo4j authentication failed for URI {NEO4J_URI}. "
        "Check NEO4J_USERNAME and NEO4J_PASSWORD in .env."
    ) from e
except ServiceUnavailable as e:
    raise RuntimeError(
        f"Neo4j service unavailable for URI {NEO4J_URI}. "
        "Check NEO4J_URI and internet/DNS access to *.databases.neo4j.io."
    ) from e

def run_query(cypher: str, params: Optional[dict] = None) -> list:
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return [record.data() for record in result]

Neo4j connected: neo4j+s://29c61e51.databases.neo4j.io


In [96]:
import json
import re

STRUCTURED_CHUNK_PROMPT = """
Extract structured information from the text chunk.
Return ONLY valid JSON in this exact format:
{{
  "summary": "",
  "key_topics": [""],
  "key_entities": [""],
  "agri_signals": [""]
}}

Rules:
- Keep summary short and factual.
- key_topics should be broad themes.
- key_entities should be specific names (crops, regions, inputs, practices).
- agri_signals should capture agriculture-specific hints (irrigation, soil, pest, yield, fertilizer, weather, etc).

Text:
{text}
"""

def extract_chunk_structure(chunk: str) -> dict:
    prompt = STRUCTURED_CHUNK_PROMPT.format(text=chunk)
    raw = ask_gemini(prompt)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return {"summary": "", "key_topics": [], "key_entities": [], "agri_signals": []}

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"summary": "", "key_topics": [], "key_entities": [], "agri_signals": []}

    return {
        "summary": str(parsed.get("summary", "")).strip(),
        "key_topics": [str(x).strip() for x in parsed.get("key_topics", []) if str(x).strip()],
        "key_entities": [str(x).strip() for x in parsed.get("key_entities", []) if str(x).strip()],
        "agri_signals": [str(x).strip() for x in parsed.get("agri_signals", []) if str(x).strip()],
    }

In [97]:
# convert raw chunks to structured chunks
graph_documents = []
for i, chunk in enumerate(chunks):
    graph_documents.append({
        "chunk_id": f"chunk-{i}",
        "text": chunk,
        "structure": extract_chunk_structure(chunk),
    })

len(graph_documents), graph_documents[0]["structure"]

(64,
 {'summary': 'Development of Site Specific Fertilizer Recommendation (SSFR) using a plant test kit for sustainable food crop production, implemented by the Department and Ministry of Agriculture in collaboration with FAO.',
  'key_topics': ['Site Specific Fertilizer Recommendation (SSFR)',
   'Sustainable Food Crop Production'],
  'key_entities': ['Plant Test Kit',
   'FAO',
   'Department of Agriculture',
   'Ministry of Agriculture',
   'TCP/SRL/3606'],
  'agri_signals': ['Fertilizer recommendation',
   'Plant test kit',
   'Food crop production']})

In [98]:
graph_documents[0]

{'chunk_id': 'chunk-0',
 'text': 'SITE SPECIFI C FERTILIZER\nRECOMMENDATION (SSFR) USING\nPLANT TEST KIT\nDevelopment of Site Specific Fertilizer Recommendation (SSFR)\nfor Sustainable Food Crop Production\n(FAO - TCP/SRL /3606)\nDepartment of Agriculture\nMinistry of Agriculture\nin colabor',
 'structure': {'summary': 'Development of Site Specific Fertilizer Recommendation (SSFR) using a plant test kit for sustainable food crop production, implemented by the Department and Ministry of Agriculture in collaboration with FAO.',
  'key_topics': ['Site Specific Fertilizer Recommendation (SSFR)',
   'Sustainable Food Crop Production'],
  'key_entities': ['Plant Test Kit',
   'FAO',
   'Department of Agriculture',
   'Ministry of Agriculture',
   'TCP/SRL/3606'],
  'agri_signals': ['Fertilizer recommendation',
   'Plant test kit',
   'Food crop production']}}

In [99]:
print("Key topics:", graph_documents[0]["structure"]["key_topics"])
print("Key entities:", graph_documents[0]["structure"]["key_entities"])

Key topics: ['Site Specific Fertilizer Recommendation (SSFR)', 'Sustainable Food Crop Production']
Key entities: ['Plant Test Kit', 'FAO', 'Department of Agriculture', 'Ministry of Agriculture', 'TCP/SRL/3606']


In [100]:
print("Summary:")
print(graph_documents[0]["structure"]["summary"])

print("Agri signals:", graph_documents[0]["structure"]["agri_signals"])

Summary:
Development of Site Specific Fertilizer Recommendation (SSFR) using a plant test kit for sustainable food crop production, implemented by the Department and Ministry of Agriculture in collaboration with FAO.
Agri signals: ['Fertilizer recommendation', 'Plant test kit', 'Food crop production']


In [104]:
# add structured chunks to graph
for doc in graph_documents:
    s = doc["structure"]
    run_query(
        """
        MERGE (d:Document {id: $doc_id})
        SET d.text = $text,
            d.summary = $summary,
            d.key_entities = $key_entities,
            d.agri_signals = $agri_signals
        WITH d
        UNWIND $key_topics AS topic
        MERGE (t:Topic {name: topic})
        MERGE (d)-[:HAS_TOPIC]->(t)
        """,
        {
            "doc_id": doc["chunk_id"],
            "text": doc["text"],
            "summary": s["summary"],
            "key_entities": s["key_entities"],
            "agri_signals": s["agri_signals"],
            "key_topics": s["key_topics"],
        },
    )

In [105]:
# indexing enables fast searches on chunk text and summary
def create_fulltext_index():
    run_query(
        "CREATE FULLTEXT INDEX chunk_index IF NOT EXISTS FOR (d:Document) ON EACH [d.text, d.summary]"
    )

create_fulltext_index()

### Querying the Graph and Entity Retrieval

In [106]:
ENTITY_EXTRACTION_PROMPT = """
Extract entities from the user question.
Return ONLY valid JSON in this format:
{{"names": ["entity1", "entity2"]}}

Question:
{question}
"""

In [107]:
def extract_entities(question: str) -> list:
    raw = ask_gemini(ENTITY_EXTRACTION_PROMPT.format(question=question))
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return []

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

    names = []
    for name in parsed.get("names", []):
        cleaned = str(name).strip()
        if cleaned:
            names.append(cleaned)
    return names

In [108]:
entity_chain = extract_entities

In [109]:
entity_chain("which crops are discussed and what farming practices are mentioned?")

[]

### Graph Retriever

Lucene special characters are sanitized before full-text search.

In [110]:
import re

def remove_lucene_chars(value: str) -> str:
    return re.sub(r"[+\-&|!(){}\[\]^\"~*?:\\/]", " ", value)

def generate_full_text_query(value: str) -> str:
    words = [word for word in remove_lucene_chars(value).split() if word]
    if not words:
        return ""
    if len(words) == 1:
        return f"{words[0]}~2"
    return " AND ".join([f"{word}~2" for word in words])

def graph_retriever(question: str) -> str:
    result = []
    entities = extract_entities(question)
    search_terms = entities if entities else [question]

    for term in search_terms:
        fulltext_query = generate_full_text_query(term)
        if not fulltext_query:
            continue

        response = run_query(
            """
            CALL db.index.fulltext.queryNodes('chunk_index', $query, {limit: 3})
            YIELD node, score
            RETURN
              node.id AS chunk_id,
              coalesce(node.summary, '') AS summary,
              coalesce(node.key_entities, []) AS key_entities,
              coalesce(node.agri_signals, []) AS agri_signals,
              score
            ORDER BY score DESC
            LIMIT 3
            """,
            {"query": fulltext_query},
        )

        for row in response:
            result.append(
                f"{row['chunk_id']} | score={row['score']:.3f} | summary={row['summary']} | entities={row['key_entities']} | agri={row['agri_signals']}"
            )

    return "\n".join(result)

In [111]:
print(graph_retriever("what are the relationships between crops, soil, and irrigation?"))

chunk-34 | score=2.563 | summary=The procedure involves adding 10 drops of nitrate N1 to a filtrate in a spot plate using a pipette, followed by adding 0.5 g of nitrate reagent 2 powder with a spoon and stirring thoroughly. | entities=['nitrate N1', 'nitrate reagent 2 powder', 'filtrate', 'spot plate', 'pipette', 'spoon', 'stirring rod'] | agri=['Nitrate testing (implying soil or water nutrient analysis)', 'Reagent', 'Filtrate']
chunk-56 | score=2.367 | summary=The text discusses potassium MOP and urea recommendations for vegetable crops, based on plant sap test results and crop age, referencing a Table 6 for application rates. | entities=['MOP (Muriate of Potash)', 'Potassium', 'Urea', 'Vegetable crops', 'Plant sap test'] | agri=['Rate of application', 'Potassium recommendation', 'Urea recommendation', 'Plant sap test results', 'Age of the crop', 'Fertilizer (MOP, Urea)']
chunk-40 | score=1.939 | summary=This section outlines the recommendation of nitrogen fertilizer for maize, emphas

### Semantic Search Retriever

In [112]:
import numpy as np

chunk_vectors = []
for chunk in chunks:
    chunk_vectors.append(
        {
            "text": chunk,
            "embedding": np.array(embed_text(chunk, task_type="retrieval_document"), dtype=float),
        }
    )

print(f"Indexed {len(chunk_vectors)} chunks for semantic search")

Indexed 64 chunks for semantic search


In [113]:
def semantic_search(question: str, k: int = 2) -> list:
    query_embedding = np.array(
        embed_text(question, task_type="retrieval_query"),
        dtype=float,
    )
    query_norm = np.linalg.norm(query_embedding) + 1e-12

    scored = []
    for item in chunk_vectors:
        doc_embedding = item["embedding"]
        score = float(
            np.dot(query_embedding, doc_embedding)
            / (query_norm * (np.linalg.norm(doc_embedding) + 1e-12))
        )
        scored.append((score, item["text"]))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [text for _, text in scored[:k]]

semantic_search("what are sustainable farming methods mentioned in the documents?", k=2)

['ghest \npriority in sustainable agriculture. Therefore, fertilizer application based on plant \ndemand is an important practice in safe use of fertilizer. Plant testing to decide its \nnutrient demand can be done at any time of the growing season. Most ',
 'ls of plant phosphorous content\n4\nFigures\nTables\n1\nSince general public is highly concerned about the increase in environmental pollution \nand fertilizer prices, efficient utilization of fertilizer should be given the highest \npriority in sustainable']

In [114]:
def retriever(question: str) -> str:
    graph_search_result = graph_retriever(question)
    semantic_search_result = semantic_search(question, k=2)
    final_data = (
        f"Graph data:\n{graph_search_result}\n\n"
        f"Text data:\n{' '.join(semantic_search_result)}"
    )
    return final_data

### Define Prompt Template for RAG

In [115]:
RAG_PROMPT = """
Answer the question using only the provided context.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}
Answer:
"""

### Create RAG Chain

In [116]:
def answer_question(question: str) -> str:
    context = retriever(question)
    prompt = RAG_PROMPT.format(context=context, question=question)
    return ask_gemini(prompt)

### Invoke RAG Chain with Example Questions

In [117]:
response = answer_question("give a fertilizers names")

print(response)

import requests
from html import unescape
from dotenv import load_dotenv

# Ensure latest values from .env are available in this cell
load_dotenv(override=True)

# Translate the latest RAG response using Google Translate API key from .env
TRANSLATE_API_KEY = (
    os.getenv("GOOGLE_TRANSLATE_API_KEY")
    or os.getenv("GOOGLE_TRANSLATE_KEY")
    or os.getenv("TRANSLATE_API_KEY")
)

if not TRANSLATE_API_KEY:
    raise ValueError(
        "Google Translate API key not found in .env. "
        "Use GOOGLE_TRANSLATE_API_KEY or GOOGLE_TRANSLATE_KEY or TRANSLATE_API_KEY."
    )

if "response" not in globals() or not str(response).strip():
    raise ValueError("No response text found. Run a question cell first to create 'response'.")

TARGET_LANG = "si"  # Change to your target language code, e.g., en, ta, hi

translate_url = "https://translation.googleapis.com/language/translate/v2"
payload = {
    "q": str(response),
    "target": TARGET_LANG,
    "format": "text",
    "key": TRANSLATE_API_KEY,
}

res = requests.post(translate_url, data=payload, timeout=30)
if not res.ok:
    try:
        api_error = res.json()
    except Exception:
        api_error = {"raw": res.text}
    raise RuntimeError(f"Google Translate API error ({res.status_code}): {api_error}")

translated_text = unescape(res.json()["data"]["translations"][0]["translatedText"])

print("\nTranslated response:\n")
print(translated_text)

Fertilizer, Nitrogen, TSP

Translated response:

පොහොර, නයිට්‍රජන්, TSP
